# Phase 01-03：本地模型部署 — Ollama 实战

> **学习目标**：安装 Ollama、拉取开源模型、理解 GGUF 量化、掌握本地模型 vs API 的选型决策

> **前置要求**：完成 `01-python-essentials.ipynb` 和 `02-llm-api-basics.ipynb`

## 1. 为什么需要本地模型？

| 维度 | API（OpenAI/Anthropic） | 本地（Ollama） |
|------|------------------------|----------------|
| 推理质量 | ★★★★★ 最新模型 | ★★★~★★★★ 开源模型 |
| 延迟 | 网络 RTT + 推理 | 纯推理（视 GPU） |
| 成本 | 按 Token 付费 | 固定硬件成本 |
| 数据隐私 | 数据离开本机 | **完全本地** |
| 离线能力 | 需要网络 | **可离线** |
| 定制化 | 不可控 | **可微调/LoRA** |
| 运维 | 零运维 | 需要 GPU 运维 |

**使用场景**：
- 🔒 敏感数据（医疗/金融/军事）不能离开本机
- 💰 高吞吐量场景，月均 Token 超 7500 万（自部署更便宜）
- 🎛️ 需要微调模型以适配特定领域术语
- 📡 离线环境或网络不稳定场景

## 2. 安装 Ollama

In [ ]:
"""
Ollama 安装与验证

安装方式：
  Windows/Mac: 从 https://ollama.com 下载安装包
  Linux: curl -fsSL https://ollama.com/install.sh | sh
"""
import subprocess
import sys

def check_ollama() -> bool:
    """检查 Ollama 是否已安装并运行"""
    try:
        result = subprocess.run(
            ["ollama", "--version"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            print(f"✅ Ollama 已安装: {result.stdout.strip()}")
            return True
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    
    print("❌ Ollama 未安装或未找到！")
    print("请访问 https://ollama.com 下载安装")
    return False

OLLAMA_AVAILABLE = check_ollama()

## 3. 拉取和运行模型

Ollama 提供两种 API 端点：
- **原生 API**：`POST http://localhost:11434/api/generate`
- **OpenAI 兼容 API**：`POST http://localhost:11434/v1/chat/completions`

In [ ]:
import requests
import json

# 推荐拉取的模型（在终端执行）
RECOMMENDED_MODELS = {
    "qwen2.5:7b": "中文最佳，阿里的通义千问 2.5",
    "qwen2.5:14b": "中文更强但需要更多显存",
    "llama3:8b": "Meta 开源，英文通用强",
    "deepseek-r1:7b": "深度求索，推理能力强",
    "nomic-embed-text": "本地 Embedding 模型（137M 参数，轻量）",
}

def ollama_chat_native(model: str, prompt: str, stream: bool = False) -> str:
    """通过 Ollama 原生 API 调用模型"""
    if not OLLAMA_AVAILABLE:
        return f"[Ollama 不可用] 将调用模型 {model}，输入: {prompt[:50]}..."
    
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream
    }
    try:
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        # Ollama native API returns one JSON per line when streaming
        if stream:
            result = ""
            for line in response.text.strip().split("\n"):
                if line:
                    result += json.loads(line).get("response", "")
            return result
        else:
            return response.json().get("response", "")
    except requests.exceptions.ConnectionError:
        return f"[连接失败] 请确保 Ollama 正在运行：在终端执行 'ollama serve'"
    except Exception as e:
        return f"[错误] {e}"

# 使用 OpenAI 兼容端点（可以复用 openai Python 库！）
try:
    from openai import OpenAI
    local_client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"  # Ollama 不验证 key，但需要传一个值
    )
    print("✅ OpenAI 兼容客户端已配置，指向本地 Ollama")
except ImportError:
    print("⚠️ openai 库未安装，跳过兼容客户端配置")
    local_client = None

print(f"\n推荐模型列表：")
for model_name, desc in RECOMMENDED_MODELS.items():
    print(f"  📦 {model_name:20s} — {desc}")

## 4. 理解 GGUF 与量化

**GGUF**（GPT-Generated Unified Format）是 llama.cpp 的统一模型格式，替代了旧的 GGML。

### 量化级别对比

| 量化级别 | 质量 | 大小（7B模型） | 显存需求 | 适用场景 |
|---------|------|--------------|---------|---------|
| Q2_K | ★★ | ~2.8 GB | 4 GB | 极限低显存 |
| Q3_K_M | ★★★ | ~3.5 GB | 6 GB | 低显存设备 |
| **Q4_K_M** | ★★★★ | ~4.5 GB | 6-8 GB | **推荐：性价比最优** |
| Q5_K_M | ★★★★ | ~5.5 GB | 8 GB | 中等设备 |
| Q8_0 | ★★★★★ | ~7.5 GB | 10 GB | 接近原始质量 |
| F16 | ★★★★★ | ~14 GB | 16 GB | 无损失（基准） |

> 💡 **经验法则**：Q4_K_M 是质量和大小之间的最佳平衡点，通常是默认推荐。

## 5. 对比不同模型

In [ ]:
import time

def benchmark_models(prompt: str, models: list[str] = None):
    """对比多个模型在同一 prompt 上的表现"""
    if models is None:
        models = ["qwen2.5:7b", "llama3:8b"]
    
    results = []
    for model in models:
        print(f"\n{'='*50}")
        print(f"🔄 模型: {model}")
        start = time.time()
        response = ollama_chat_native(model, prompt)
        latency = time.time() - start
        
        # 粗略 Token 估算（中文 ~1.5 char/token，英文 ~0.25 char/token）
        estimated_tokens = len(response) // 2  # 简化估算
        
        results.append({
            "model": model,
            "latency_s": round(latency, 2),
            "response_len": len(response),
            "est_tokens": estimated_tokens,
            "response": response[:300]
        })
        print(f"   延迟: {latency:.2f}s | 响应长度: {len(response)} 字符")
        print(f"   回复预览: {response[:200]}...")
    
    return results

# 运行基准测试
TEST_PROMPT = "请用三句话解释什么是向量数据库，并举例说明其应用场景。"
benchmark_results = benchmark_models(TEST_PROMPT)

## 6. 创建自定义模型（Modelfile）

Ollama 允许通过 Modelfile 自定义模型行为——设置 System Prompt、temperature、top_p 等。

In [ ]:
def create_custom_model():
    """创建自定义 Modelfile 并构建模型"""
    modelfile_content = """
FROM qwen2.5:7b

# 设置系统提示词
SYSTEM """你是一个专业的中英翻译助手。
规则：
1. 中文输入 → 翻译成自然流畅的英文
2. 英文输入 → 翻译成自然流畅的中文  
3. 保持原文的语气和风格
4. 专业术语使用行业标准翻译
5. 只输出翻译结果，不添加解释"""

# 调整参数
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "---"
PARAMETER stop "用户:"
"""
    
    if not OLLAMA_AVAILABLE:
        print("⚠️ Ollama 不可用，以下为模拟演示：")
        print(modelfile_content)
        print("\n在终端执行: ollama create translator -f Modelfile")
        return
    
    # 写入 Modelfile
    with open("Modelfile", "w", encoding="utf-8") as f:
        f.write(modelfile_content)
    
    print("📝 Modelfile 已创建，内容如下：")
    print(modelfile_content)
    print("\n⚠️ 请在终端执行以下命令来构建模型：")
    print("  ollama create translator -f Modelfile")
    print("  ollama run translator")

create_custom_model()

## 7. 动手练习：搭建中英翻译器

In [ ]:
def translation_exercise():
    """练习：使用 Ollama 构建中英翻译器"""
    test_sentences = [
        ("zh", "人工智能正在改变我们与计算机交互的方式。"),
        ("zh", "检索增强生成是提高大语言模型事实准确性的关键技术。"),
        ("en", "Retrieval-Augmented Generation significantly reduces hallucination in LLMs."),
        ("en", "Vector databases enable semantic search at billion-scale."),
        ("zh", "微调模型需要高质量的训练数据和合适的超参数配置。"),
    ]
    
    translation_prompt = """请将以下文本翻译成{target_lang}，只输出翻译结果：

{text}"""
    
    print("🌐 中英翻译测试")
    print("=" * 50)
    
    for lang, text in test_sentences:
        target = "英文" if lang == "zh" else "中文"
        prompt = translation_prompt.format(target_lang=target, text=text)
        
        print(f"\n📝 原文({lang}): {text}")
        result = ollama_chat_native("qwen2.5:7b", prompt)
        print(f"🌍 翻译: {result[:200]}")

translation_exercise()

## 8. 本地模型 vs API：决策指南

```
你的场景是什么？
│
├─ 数据包含敏感信息（医疗/金融/PII）？
│  └─ YES → 本地模型（数据不出本机）
│
├─ 月均 Token 消耗 > 7500 万？
│  └─ YES → 本地模型（自部署更便宜）
│
├─ 需要离线运行？
│  └─ YES → 本地模型
│
├─ 只有 CPU 没有 GPU？
│  └─ YES → API（CPU 推理太慢）
│
├─ 需要最新最强的模型（GPT-4o/Claude Opus）？
│  └─ YES → API（开源模型有差距）
│
├─ 快速原型开发，一周内上线？
│  └─ YES → API（零运维）
│
└─ 混合策略（推荐！）：
   ├─ 敏感数据 → 本地 qwen2.5/bge-m3
   ├─ 通用查询 → gpt-4o-mini
   └─ 复杂推理 → gpt-4o / claude-sonnet-4-6
```

### 自部署盈亏平衡点

| 月均 Embedding Token | 建议方案 |
|---------------------|---------|
| < 1000 万 | API（text-embedding-3-small） |
| 1000 万 - 7500 万 | API + 缓存优化 |
| **> 7500 万** | **自部署 BGE-M3** 更便宜 |

## 9. ✅ 学习检验

- [ ] 能安装 Ollama 并拉取至少 2 个模型
- [ ] 能用 Python 调用 Ollama API 获取回复
- [ ] 能解释 GGUF 量化级别及其质量/大小权衡
- [ ] 能对比至少 2 个模型在同一个 prompt 上的表现
- [ ] 能创建 Modelfile 自定义模型
- [ ] 能独立完成中英翻译器练习
- [ ] 能在 4 个场景中正确选择本地模型 vs API

> 📚 **下一步**：完成 `04-environment-config.py` 学习生产级配置管理，然后进入 [Phase 02：Prompt 工程](../phase-02-Prompt工程/)